# Create Dreyfus Foundation Awards (RESEARCH PATTERN, multi-scheme HTML scrape)

Camille and Henry Dreyfus Foundation is a US chemistry-focused philanthropic
funder (established 1946). Its annual award announcements are published as
per-year HTML pages on www.dreyfus.org and enumerable through the
foundation's own WordPress post-sitemap. Each page lists a single program's
recipients for one year, with the awardee name (in `<strong>`/`<b>`), host
institution (inline text), and project title (in `<em>`/`<i>`).

**Schemes ingested in this pipeline (4):**
- Camille Dreyfus Teacher-Scholar Awards (2016-2026, USD 100,000 per recipient)
- Henry Dreyfus Teacher-Scholar Awards (2016-2025, USD 75,000 per recipient)
- Supplemental Grants for Teacher-Scholars (2025, amount NULL per source)
- Machine Learning in the Chemical Sciences and Engineering Awards (2022 one-off, amount NULL per source)

**Awarding body:** Camille and Henry Dreyfus Foundation - F4320306315 (US, DOI 10.13039/100001082)

**Schema choices:**
- One row per (scheme, year, name) triple. `funder_award_id` = `dreyfus-{scheme}-{year}-{name_slug}`.
- `funder_scheme` = the program's official display name (e.g. `Camille Dreyfus Teacher-Scholar Awards`), populated upstream in the source script.
- `funding_type` = `research` for all 4 schemes.
- `amount` / `currency`: Camille and Henry rows carry the official per-recipient amount from each program's landing page; Supplemental and ML 2022 rows ship NULL because the source pages don't publish per-recipient amounts (the program-level total IS published but isn't apportioned).
- `lead_investigator.affiliation.country`: NULL by source authority. Dreyfus eligibility is US + Canadian academic institutions; the source pages don't print country and we don't infer it from institution name.
- No `declined` column; the source has no concept of declined recipients.
- `start_year` = announcement year; `end_year` / `start_date` / `end_date` NULL (year-only granularity on source).

**Source:** https://www.dreyfus.org/post-sitemap.xml + per year-program announcement pages

**S3 location:** `s3a://openalex-ingest/awards/dreyfus/dreyfus_awardees.parquet`

**Method:** WordPress site, but WP REST API is restricted by Kadence Security (401). Scraper enumerates per-year announcement URLs from the public `/post-sitemap.xml`, then parses rendered HTML (handles both modern `<strong>`/`<em>` and 2019 heritage `<b>`/`<i>` markup variants). New precedent for HTML-scrape of a WordPress site where WP REST is gated.

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.dreyfus_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/dreyfus/dreyfus_awardees.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.dreyfus_raw;

## Step 1.5: Inspect raw + scheme/year/amount distribution

Expected: ~272 rows across 4 schemes, year range 2016-2026, ~91% with USD amount (Camille + Henry), ~9% NULL (Supplemental + ML, §6.7-waived schemes).

In [ ]:
%sql
DESCRIBE openalex.awards.dreyfus_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.dreyfus_raw LIMIT 5;

In [ ]:
%sql
-- Row counts per (scheme, year) to confirm sitemap enumeration matches expectation.
SELECT scheme, year, COUNT(*) AS rows,
       COUNT(institution) AS has_inst,
       COUNT(research_title) AS has_title,
       COUNT(amount) AS has_amount
FROM openalex.awards.dreyfus_raw
GROUP BY scheme, year
ORDER BY scheme, year;

In [ ]:
%sql
-- Amount coverage by scheme. Camille + Henry must be 100% (official per-recipient amounts);
-- Supplemental + ML must be 0% (no per-recipient amount published, §6.7-waived).
SELECT scheme,
       COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
       collect_set(currency) AS currencies
FROM openalex.awards.dreyfus_raw
GROUP BY scheme
ORDER BY scheme;

In [ ]:
%sql
-- Top institutions (sanity-check parsing didn't bleed titles into the institution field).
SELECT institution, COUNT(*) AS rows
FROM openalex.awards.dreyfus_raw
WHERE institution IS NOT NULL
GROUP BY institution
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast - verify Dreyfus Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306315;

## Step 2: Transform to award schema

Per-row `display_name` is constructed from the scheme label + recipient name (e.g. `Camille Dreyfus Teacher-Scholar Awards - Andrew Boydston`). The recipient's research project title (the `<em>` content from the source page) becomes `description`.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.dreyfus_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306315  -- Camille and Henry Dreyfus Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(n.scheme_label, ' - ', n.name) AS display_name,
    n.research_title AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    n.funding_type,
    n.scheme_label AS funder_scheme,
    'dreyfus_foundation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                CAST(NULL AS STRING) AS country,  -- Dreyfus eligibility is US+CA; not inferred from institution name
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.dreyfus_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 131

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dreyfus_foundation' AND priority = 131;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    131 AS priority  -- Dreyfus Foundation
FROM openalex.awards.dreyfus_awards;

## Step 6: Verification

§6.7 amount-coverage check is **PARTIALLY WAIVED**: full coverage required on Camille + Henry schemes (where Dreyfus publishes official per-recipient amounts); waived on Supplemental + ML 2022 (where Dreyfus does not). Expect overall ~91% amount coverage.

In [ ]:
%sql
SELECT COUNT(*) AS total_dreyfus_rows FROM openalex.awards.dreyfus_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.dreyfus_awards;

In [ ]:
%sql
-- §6.3 Data completeness, overall corpus.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.dreyfus_awards;

In [ ]:
%sql
-- §6.7 split by scheme. Camille and Henry must hit pct_amount = 100;
-- Supplemental + Machine Learning must show pct_amount = 0 (waived).
SELECT funder_scheme,
       COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
       collect_set(currency) AS currencies
FROM openalex.awards.dreyfus_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Year range and per-year counts (announcement year, not grant period).
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.dreyfus_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct sanity - all rows must point to the Dreyfus funder row.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.dreyfus_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS program, funding_type, start_year, amount, currency,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS institution
FROM openalex.awards.dreyfus_awards
ORDER BY start_year DESC, family
LIMIT 10;